In [2]:
!pip install -q gemmi mdanalysis rdkit biopython pandas numpy matplotlib plotly pubchempy gradio

In [3]:
import os, json
from typing import Optional, Dict, Tuple, Any

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

import gemmi
import MDAnalysis as mda
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from IPython.display import display
import Bio.PDB

# Optional: PubChem fallback
try:
    import pubchempy as pcp
except Exception:
    pcp = None

In [4]:
def default_cfg() -> Dict[str, Any]:
    return {
        "cache_dir": "./cache_bio",
        "pdb_dir": "./pdb_structures",
        "timeout_s": 60,
        "user_agent": "BioPipeline/1.0 (research; contact: none)",
        "ccd_smiles_url": "https://files.wwpdb.org/pub/pdb/data/monomers/Components-smiles-stereo-oe.smi",
        "ligand_cif_url_tpl": "https://files.rcsb.org/ligands/download/{ligand_id}.cif",
        "pdb_cif_url_tpl": "https://files.rcsb.org/download/{pdb_id}.cif",
    }


In [5]:
def ensure_dir(path: str) -> str:
    """Checks the folder; if it does not exist, creates it and returns the path."""
    os.makedirs(path, exist_ok=True)
    return path

def http_get_text(url: str, cfg: Dict[str, Any]) -> str:
    """Downloads text (str) from a URL (using requests)."""
    headers = {"User-Agent": cfg["user_agent"]}
    r = requests.get(url, timeout=cfg["timeout_s"], headers=headers)
    r.raise_for_status()
    return r.text

def cache_file(cfg: Dict[str, Any], filename: str) -> str:
    """Constructs the full path of the cache file."""
    ensure_dir(cfg["cache_dir"])
    return os.path.join(cfg["cache_dir"], filename)

def save_text(path: str, text: str) -> None:
    """Writes text to a file."""
    ensure_dir(os.path.dirname(path) or ".")
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def save_json(path: str, obj: Any) -> None:
    """Writes JSON to a file."""
    ensure_dir(os.path.dirname(path) or ".")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)

def load_json(path: str) -> Any:
    """Reads JSON from a file."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


In [6]:
def load_ccd_smiles_bulk(cfg: Dict[str, Any], force: bool = False) -> Dict[str, str]:
    """
    Downloads the wwPDB Components-smiles file once and caches it.
    Returns: {CCD_ID -> SMILES}
    """
    cpath = cache_file(cfg, "ccd_smiles_bulk.json")

    if (not force) and os.path.exists(cpath):
        d = load_json(cpath)
        print(f"CCD bulk SMILES loaded from cache: {len(d):,} ligand")
        return d

    print("Loading CCD bulk SMILES (wwPDB)...")
    text = http_get_text(cfg["ccd_smiles_url"], cfg)

    d: Dict[str, str] = {}
    for line in text.splitlines():
        line = line.strip()
        if (not line) or line.startswith("#"):
            continue
        parts = [p.strip() for p in line.split("\t") if p.strip()]
        if len(parts) >= 2:
            smiles, ccd_id = parts[0], parts[1].upper()
            if 1 <= len(ccd_id) <= 5 and ccd_id.replace(" ", "").isalnum():
                d[ccd_id] = smiles

    save_json(cpath, d)
    print(f" Cache written: {cpath} | {len(d):,} ligand")
    return d

In [7]:
def download_pdb_cif(pdb_id: str, cfg: Dict[str, Any], force: bool = False) -> str:
    """
    Downloads the PDB mmCIF file from RCSB and saves it to disk.
    Returns: the path of the saved .cif file.
    """
    pdb_id = pdb_id.strip().upper()
    ensure_dir(cfg["pdb_dir"])
    out_path = os.path.join(cfg["pdb_dir"], f"{pdb_id}.cif")

    if os.path.exists(out_path) and not force:
        print(f"PDB mmCIF cache: {out_path}")
        return out_path

    url = cfg["pdb_cif_url_tpl"].format(pdb_id=pdb_id)
    print(f" PDB mmCIF downloaded: {pdb_id}")
    text = http_get_text(url, cfg)
    save_text(out_path, text)
    print(f" Saved: {out_path}")
    return out_path

def load_universe_from_cif(cif_path: str) -> mda.Universe:
    """
    MDAnalysis may have issues reading mmCIF files in some environments.
    Workaround: read with gemmi → write a temporary PDB file → read the PDB with MDAnalysis.
    """
    structure = gemmi.read_structure(cif_path)
    tmp_pdb = cif_path.replace(".cif", "._tmp.pdb")
    structure.write_pdb(tmp_pdb)

    u = mda.Universe(tmp_pdb, format="PDB")

    try:
        os.remove(tmp_pdb)
    except Exception:
        pass

    return u

In [8]:
def analyze_protein_and_ligand(u: mda.Universe, ligand_resname: str) -> Dict[str, Any]:
    """
    Number of protein atoms, Rg (radius of gyration),
    number of ligand atoms, and the distance between the protein COM and ligand COM.
    """
    ligand_resname = ligand_resname.strip().upper()

    protein = u.select_atoms("protein")
    rg = float(protein.radius_of_gyration()) if len(protein) else np.nan

    ligand = u.select_atoms(f"resname {ligand_resname}")
    dist = None
    if len(protein) and len(ligand):
        dist = float(np.linalg.norm(protein.center_of_mass() - ligand.center_of_mass()))

    return {
        "protein_atoms": int(len(protein)),
        "protein_rg_A": None if np.isnan(rg) else round(rg, 3),
        "ligand_atoms": int(len(ligand)),
        "prot_lig_distance_A": None if dist is None else round(dist, 3),
    }


In [9]:
   def parse_smiles_from_ligand_cif(cif_text: str) -> Optional[str]:
    """
    Tries to find SMILES inside the ligand CIF (using the gemmi CIF parser).
    Checks both old and new loop names.
    """
    doc = gemmi.cif.read_string(cif_text)
    block = doc.sole_block()

    loop_candidates = ["_pdbx_chem_comp_descriptor", "_chem_comp_descriptor"]
    for loop_name in loop_candidates:
        loop = block.find_loop(loop_name)
        if not loop:
            continue

        tags = list(loop.tags)
        type_idx = value_idx = None

        for i, tag in enumerate(tags):
            t = tag.lower()
            if t.endswith(".type") or t.endswith(".descriptor_type"):
                type_idx = i
            if t.endswith(".descriptor") or t.endswith(".value"):
                value_idx = i

        if type_idx is None or value_idx is None:
            continue

        for row in loop:
            d_type = (str(row[type_idx]) if row[type_idx] else "").upper()
            if "SMILES" in d_type:
                val = (str(row[value_idx]) if row[value_idx] else "").strip()
                if val:
                    return val
    return None

def fetch_ligand_smiles(
    ligand_id: str,
    cfg: Dict[str, Any],
    bulk_smiles: Optional[Dict[str, str]] = None
) -> Dict[str, Any]:
    """
    SMILES retrieval strategy:
    - Bulk CCD dictionary (fastest)
    - Download RCSB ligand CIF + parse it
    - PubChem fallback (if pubchempy is available)
    """
    ligand_id = ligand_id.strip().upper()

    # 1) bulk
    if bulk_smiles and ligand_id in bulk_smiles:
        return {"ligand_id": ligand_id, "smiles": bulk_smiles[ligand_id], "source": "CCD_BULK"}

    # 2) ligand CIF
    url = cfg["ligand_cif_url_tpl"].format(ligand_id=ligand_id)
    try:
        cif_text = http_get_text(url, cfg)
        smiles = parse_smiles_from_ligand_cif(cif_text)
        if smiles:
            return {"ligand_id": ligand_id, "smiles": smiles, "source": "CCD_LIGAND_CIF"}
    except Exception:
        pass

    # 3) PubChem
    if pcp is not None:
        try:
            comps = pcp.get_compounds(ligand_id, "name")
            if comps and comps[0].canonical_smiles:
                return {"ligand_id": ligand_id, "smiles": comps[0].canonical_smiles, "source": "PUBCHEM"}
        except Exception:
            pass

    return {"ligand_id": ligand_id, "smiles": None, "source": None}



In [10]:
def compute_rdkit_descriptors(smiles: str) -> Tuple[Optional[Dict[str, Any]], Optional[Chem.Mol]]:
    """SMILES → RDKit Mol → selected descriptors"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {"error": "Invalid SMILES"}, None

    desc = {
        "MolWt": round(Descriptors.MolWt(mol), 2),
        "LogP": round(Descriptors.MolLogP(mol), 2),
        "HDonors": int(Descriptors.NumHDonors(mol)),
        "HAcceptors": int(Descriptors.NumHAcceptors(mol)),
        "RotBonds": int(Descriptors.NumRotatableBonds(mol)),
        "TPSA": round(Descriptors.TPSA(mol), 2),
    }
    return desc, mol

def draw_ligand_2d(mol: Chem.Mol, title: str = "", size: Tuple[int, int] = (420, 420)):
    """2D image of the ligand using RDKit"""
    img = Draw.MolToImage(mol, size=size)
    if title:
        print(f"Ligand 2D: {title}")
    display(img)
    return img

def plot_desc_bar(desc: Dict[str, Any]):
    """Descriptor bar plot using Matplotlib"""
    labels = list(desc.keys())
    vals = list(desc.values())
    plt.figure(figsize=(10, 4))
    plt.bar(labels, vals)
    plt.title("RDKit Molecular Descriptors")
    plt.ylabel("Value")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [11]:
def get_sequence_and_aa_freq(cif_path: str) -> Tuple[str, pd.DataFrame]:
    """
    Extracts the protein sequence from mmCIF (using PPBuilder).
    Then returns the amino acid frequency table.
    """
    parser = Bio.PDB.MMCIFParser(QUIET=True)
    structure = parser.get_structure("pdb", cif_path)
    ppb = Bio.PDB.PPBuilder()

    sequence = "".join(str(pp.get_sequence()) for pp in ppb.build_peptides(structure))
    if not sequence:
        return "", pd.DataFrame(columns=["AminoAcid", "Count", "Frequency"])

    aa_list = list(sequence)
    unique, counts = np.unique(aa_list, return_counts=True)
    df = pd.DataFrame({"AminoAcid": unique, "Count": counts})
    df["Frequency"] = df["Count"] / len(sequence)
    df = df.sort_values("Frequency", ascending=False).reset_index(drop=True)
    return sequence, df

def plot_aa_frequency_top10(freq_df: pd.DataFrame):
    """Top-10 amino acid frequency plot using Plotly"""
    if freq_df.empty:
        print("Sequence / frequency is empty.")
        return None

    top10 = freq_df.head(10)
    fig = px.bar(top10, x="AminoAcid", y="Frequency",
                 title="Top 10 Amino Acids (Frequency)", text="Frequency")
    fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
    fig.show()
    return fig

In [12]:
def run_pipeline(pdb_id: str, ligand_resname: str, cfg: Dict[str, Any], show: bool = True) -> Dict[str, Any]:
    """
    Runs all stages sequentially and returns the results.
    If show=True, it also displays the DataFrame and plots.
    """
    pdb_id = pdb_id.strip().upper()
    ligand_resname = ligand_resname.strip().upper()

    bulk_smiles = load_ccd_smiles_bulk(cfg)
    cif_path = download_pdb_cif(pdb_id, cfg)
    u = load_universe_from_cif(cif_path)

    prot_stats = analyze_protein_and_ligand(u, ligand_resname)
    smiles_info = fetch_ligand_smiles(ligand_resname, cfg, bulk_smiles=bulk_smiles)

    desc, mol = (None, None)
    if smiles_info["smiles"]:
        desc, mol = compute_rdkit_descriptors(smiles_info["smiles"])

    seq, freq_df = get_sequence_and_aa_freq(cif_path)

    summary = {
        "pdb_id": pdb_id,
        "ligand": ligand_resname,
        "protein_atoms": prot_stats["protein_atoms"],
        "ligand_atoms": prot_stats["ligand_atoms"],
        "protein_rg_A": prot_stats["protein_rg_A"],
        "prot_lig_distance_A": prot_stats["prot_lig_distance_A"],
        "smiles": smiles_info["smiles"],
        "smiles_source": smiles_info["source"],
        "sequence_length": len(seq),
    }

    if show:
        print("Pipeline finished.\n")
        display(pd.DataFrame([summary]))

        if mol is not None and desc is not None and "error" not in desc:
            draw_ligand_2d(mol, title=ligand_resname)
            plot_desc_bar(desc)
        else:
            print("RDKit step skipped (SMILES not found or invalid).")
        if seq:
            print(f"Sequence length: {len(seq)}")
            display(freq_df.head(5))
            plot_aa_frequency_top10(freq_df)
        else:
            print("Sequence extraction returned empty.")

    return {
        "summary": summary,
        "rdkit_descriptors": desc,
        "aa_freq_df": freq_df,
        "sequence": seq,
        "ligand_mol": mol,
    }

In [13]:

import gradio as gr

def gradio_analyze(pdb_id: str, ligand_resname: str):
    """Wrapper for Gradio: run_pipeline(show=False) → markdown + image + plot."""
    if not pdb_id or not ligand_resname:
        return "Enter PDB ID and ligand resname", None, None

    cfg = default_cfg()
    out = run_pipeline(pdb_id, ligand_resname, cfg, show=False)

    s = out["summary"]
    freq_df = out["aa_freq_df"]
    mol = out["ligand_mol"]

    md = f"""
### Analysis result
- **PDB ID:** {s["pdb_id"]}
- **Ligand:** {s["ligand"]}
- **Protein atoms:** {s["protein_atoms"]}
- **Ligand atoms:** {s["ligand_atoms"]}
- **Rg:** {s["protein_rg_A"] if s["protein_rg_A"] is not None else "N/A"} Å
- **Protein–Ligand distance:** {s["prot_lig_distance_A"] if s["prot_lig_distance_A"] is not None else "N/A"} Å
- **Sequence length:** {s["sequence_length"]}
- **SMILES source:** {s["smiles_source"] if s["smiles_source"] else "N/A"}
- **SMILES:** {(s["smiles"][:120] + "…") if s["smiles"] else "Not found"}
"""

    ligand_img = Draw.MolToImage(mol, size=(420, 420)) if mol is not None else None

    aa_fig = None
    if not freq_df.empty:
        top10 = freq_df.head(10)
        aa_fig = px.bar(top10, x="AminoAcid", y="Frequency", title="Top 10 Amino Acids", text="Frequency")
        aa_fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")

    return md, ligand_img, aa_fig

def launch_gradio():
    """Starts the Gradio interface"""
    demo = gr.Interface(
        fn=gradio_analyze,
        inputs=[
            gr.Textbox(label="PDB ID", value="1AKE", placeholder="məsələn: 1AKE"),
            gr.Textbox(label="Ligand resname", value="AP5", placeholder="məsələn: ATP, HEM, NAD, ..."),
        ],
        outputs=[
            gr.Markdown(label="Results"),
            gr.Image(label="Ligand 2D (RDKit)"),
            gr.Plot(label="Amino acid frequency (Top 10)"),
        ],
        title="Protein–Ligand Analysis (Bio Pipeline)",
        description="PDB mmCIF → protein/ligand stats → ligand SMILES → RDKit descriptors → sequence & AA frequency",
        examples=[["1AKE", "AP5"], ["1GY3", "ATP"], ["1MBN", "HEM"], ["1RG7", "MTX"], ["1T2F", "NAD"]],
        allow_flagging="never",
    )
    demo.launch(share=True, debug=True)

In [ ]:

# Usage

if __name__ == "__main__":
    launch_gradio()

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2fc3ec530b2d457e6c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Loading CCD bulk SMILES (wwPDB)...
 Cache written: ./cache_bio/ccd_smiles_bulk.json | 49,803 ligand
 PDB mmCIF downloaded: 1AKE
 Saved: ./pdb_structures/1AKE.cif
